# Module 4: Train MobileNetV3 Target

This notebook is the first split-out stage of Module 4. It trains the clean MobileNetV3 target with centralized supervised learning and saves the V3 checkpoint consumed by the attack notebook.

Run this notebook from the `4_Adversarial_FL/` directory.


## Stage Goal

Establish a clean centralized MobileNetV3 target before introducing adversarial examples or federated clients. The handoff artifact is the configured target checkpoint, usually `artifacts/module4_v3_target.pt`, which `attack_module.ipynb` uses for transfer evaluation and as the initialization for clean-vs-attacked FedAvg.


## 1. Notebook Setup

Load configuration, PyTorch, the shared dataset/evaluation helpers, and the MobileNetV3 model. This target notebook does not construct a federated server.


In [ ]:
from copy import deepcopy
from pathlib import Path

import json
import yaml
import numpy as np
import torch
import matplotlib.pyplot as plt
from torch.utils.data import DataLoader

from model import MobileNetV3Transfer
from util_functions import create_data, evaluate_fn, set_seed


## 2. Configuration and Artifact Setup

This cell loads `config.yaml`, resolves the device, validates centralized target-training settings, and writes the config snapshot used by downstream notebooks.


In [ ]:
CONFIG_PATH = Path("config.yaml")
if not CONFIG_PATH.exists():
    raise FileNotFoundError("Could not locate config.yaml in the working directory")
with CONFIG_PATH.open() as f:
    CONFIG = yaml.safe_load(f)


def _merge_dicts(base: dict | None, override: dict | None) -> dict:
    merged = deepcopy(base or {})
    for key, value in (override or {}).items():
        if isinstance(value, dict) and isinstance(merged.get(key), dict):
            merged[key] = _merge_dicts(merged[key], value)
        else:
            merged[key] = deepcopy(value)
    return merged


global_config = deepcopy(CONFIG.get("global_config", {}))
data_config = deepcopy(CONFIG.get("data_config", {}))
model_config = deepcopy(CONFIG.get("model_config", {}))
alg_configs = deepcopy(CONFIG.get("algorithms", {}))
artifact_config = deepcopy(CONFIG.get("artifacts", {}))
target_training_config = deepcopy(CONFIG.get("target_training", {}))

DEFAULT_FED_CONFIG = deepcopy(alg_configs.get("FedAvg", {}).get("fed_config", {}))
TARGET_CFG = target_training_config
target_optimizer_config = TARGET_CFG.get("optimizer", {}) if isinstance(TARGET_CFG.get("optimizer", {}), dict) else {}
target_scheduler_config = TARGET_CFG.get("scheduler", {}) if isinstance(TARGET_CFG.get("scheduler", {}), dict) else {}
TARGET_BATCH_SIZE = int(TARGET_CFG.get("batch_size", DEFAULT_FED_CONFIG.get("batch_size", 64)))
TARGET_EVAL_BATCH_SIZE = int(TARGET_CFG.get("eval_batch_size", 512))
TARGET_EPOCHS = int(TARGET_CFG.get("num_epochs", TARGET_CFG.get("epochs", DEFAULT_FED_CONFIG.get("num_epochs", 3))))
NUM_CLASSES = int(model_config.get("kwargs", {}).get("num_classes", 10))
set_seed(global_config.get("seed", 42))


def get_device(preferred: str | None = None) -> torch.device:
    choice = preferred if preferred is not None else global_config.get("device")
    if isinstance(choice, str):
        if choice.startswith("cuda") and not torch.cuda.is_available():
            choice = "cpu"
        return torch.device(choice)
    return torch.device("cuda" if torch.cuda.is_available() else "cpu")


DEVICE = get_device()
global_config["device"] = str(DEVICE)

ARTIFACT_DIR = Path(artifact_config.get("dir", "artifacts"))
ARTIFACT_DIR.mkdir(parents=True, exist_ok=True)


def artifact_path(key: str, default_filename: str) -> Path:
    return ARTIFACT_DIR / artifact_config.get(key, default_filename)


def validate_target_config() -> None:
    issues = []
    optimizer_name = str(target_optimizer_config.get("name", "sgd")).lower()
    scheduler_name = str(target_scheduler_config.get("name", "none")).lower()
    if not data_config.get("dataset_path"):
        issues.append("data_config.dataset_path is required for centralized target training.")
    if not data_config.get("dataset_name"):
        issues.append("data_config.dataset_name is required for centralized target training.")
    if model_config.get("name") != "MobileNetV3Transfer":
        issues.append("model_config.name should be MobileNetV3Transfer for train_v3.ipynb.")
    if TARGET_EPOCHS <= 0:
        issues.append("target_training.num_epochs must be positive.")
    if TARGET_BATCH_SIZE <= 0:
        issues.append("target_training.batch_size must be positive.")
    if optimizer_name not in {"sgd", "local_sgd", "adam", "adamw"}:
        issues.append("target_training.optimizer.name must be one of: sgd, local_sgd, adam, adamw.")
    if scheduler_name not in {"none", "cosine", "plateau", "reduce_on_plateau"}:
        issues.append("target_training.scheduler.name must be one of: none, cosine, plateau, reduce_on_plateau.")
    if issues:
        raise ValueError("Target-training config validation failed:\n" + "\n".join(issues))

    lr = target_optimizer_config.get("lr", target_optimizer_config.get("learning_rate", DEFAULT_FED_CONFIG.get("local_stepsize", 1e-3)))
    print(
        "Centralized target config ready: "
        f"device={DEVICE}, dataset={data_config.get('dataset_name')}, epochs={TARGET_EPOCHS}, "
        f"batch_size={TARGET_BATCH_SIZE}, optimizer={optimizer_name}, lr={float(lr):.3g}"
    )


validate_target_config()

config_snapshot = deepcopy(CONFIG)
config_snapshot.setdefault("global_config", {})["resolved_device"] = str(DEVICE)
config_snapshot.setdefault("target_training", {})["resolved_mode"] = "centralized"
config_path = artifact_path("config_snapshot", "module4_config_used.json")
with config_path.open("w") as f:
    json.dump(config_snapshot, f, indent=2)

print("Loaded config from", CONFIG_PATH.resolve())
print(f"Saved config snapshot to {config_path.resolve()}")


## 3. Centralized Target Data

Load the configured dataset once and build standard train/test loaders. These loaders are not split by client.


In [ ]:
def build_target_loaders() -> tuple[DataLoader, DataLoader]:
    train_dataset, test_dataset = create_data(
        data_config["dataset_path"],
        data_config["dataset_name"],
    )
    num_workers = int(TARGET_CFG.get("num_workers", 0))
    pin_memory = bool(TARGET_CFG.get("pin_memory", DEVICE.type == "cuda"))
    train_loader = DataLoader(
        train_dataset,
        batch_size=TARGET_BATCH_SIZE,
        shuffle=True,
        drop_last=False,
        num_workers=num_workers,
        pin_memory=pin_memory,
    )
    test_loader = DataLoader(
        test_dataset,
        batch_size=TARGET_EVAL_BATCH_SIZE,
        shuffle=False,
        drop_last=False,
        num_workers=num_workers,
        pin_memory=pin_memory,
    )
    return train_loader, test_loader


target_train_loader, target_test_loader = build_target_loaders()
print(
    f"Centralized target data ready: train_examples={len(target_train_loader.dataset)}, "
    f"test_examples={len(target_test_loader.dataset)}"
)


## 4. Target Training Helpers

Define the MobileNetV3 model, loss, optimizer, scheduler, and training loop used for centralized target training.


In [ ]:
def build_target_model(pretrained: bool | None = None) -> torch.nn.Module:
    kwargs = deepcopy(model_config.get("kwargs", {}))
    kwargs["num_classes"] = NUM_CLASSES
    if pretrained is not None:
        kwargs["pretrained"] = pretrained
    model = MobileNetV3Transfer(**kwargs).to(DEVICE)
    if TARGET_CFG.get("freeze_backbone", False) and hasattr(model, "v3model"):
        for param in model.v3model.features.parameters():
            param.requires_grad = False
    return model


def build_target_criterion() -> torch.nn.Module:
    criterion_cfg = TARGET_CFG.get("criterion", {}) if isinstance(TARGET_CFG.get("criterion", {}), dict) else {}
    label_smoothing = float(criterion_cfg.get("label_smoothing", TARGET_CFG.get("label_smoothing", 0.0)))
    return torch.nn.CrossEntropyLoss(label_smoothing=label_smoothing).to(DEVICE)


def build_target_optimizer(model: torch.nn.Module) -> torch.optim.Optimizer:
    name = str(target_optimizer_config.get("name", "sgd")).lower()
    if name == "local_sgd":
        name = "sgd"
    lr = float(target_optimizer_config.get("lr", target_optimizer_config.get("learning_rate", DEFAULT_FED_CONFIG.get("local_stepsize", 1e-3))))
    weight_decay = float(target_optimizer_config.get("weight_decay", 0.0))
    trainable_params = [p for p in model.parameters() if p.requires_grad]
    if not trainable_params:
        raise RuntimeError("No trainable parameters available for target optimisation.")

    if name == "adamw":
        return torch.optim.AdamW(trainable_params, lr=lr, weight_decay=weight_decay)
    if name == "adam":
        return torch.optim.Adam(trainable_params, lr=lr, weight_decay=weight_decay)
    if name == "sgd":
        momentum = float(target_optimizer_config.get("momentum", 0.0))
        return torch.optim.SGD(trainable_params, lr=lr, momentum=momentum, weight_decay=weight_decay)
    raise ValueError(f"Unsupported target optimizer: {name}")


def build_target_scheduler(optimizer: torch.optim.Optimizer, epochs: int):
    name = str(target_scheduler_config.get("name", "none")).lower()
    if name == "none":
        return None
    if name == "cosine":
        min_lr = float(target_scheduler_config.get("min_lr", target_scheduler_config.get("eta_min", 0.0)))
        return torch.optim.lr_scheduler.CosineAnnealingLR(
            optimizer,
            T_max=max(int(epochs), 1),
            eta_min=min_lr,
        )
    if name in {"plateau", "reduce_on_plateau"}:
        return torch.optim.lr_scheduler.ReduceLROnPlateau(
            optimizer,
            mode="min",
            factor=float(target_scheduler_config.get("factor", 0.5)),
            patience=int(target_scheduler_config.get("patience", 1)),
        )
    raise ValueError(f"Unsupported target scheduler: {name}")


def _step_target_scheduler(scheduler, val_loss: float) -> None:
    if scheduler is None:
        return
    if isinstance(scheduler, torch.optim.lr_scheduler.ReduceLROnPlateau):
        scheduler.step(val_loss)
    else:
        scheduler.step()


def train_target_model(num_epochs: int | None = None):
    set_seed(global_config.get("seed", 42))
    model = build_target_model().to(DEVICE)
    criterion = build_target_criterion()
    optimizer = build_target_optimizer(model)
    epochs = int(num_epochs or TARGET_EPOCHS)
    scheduler = build_target_scheduler(optimizer, epochs)
    patience = int(TARGET_CFG.get("early_stop_patience", 0))
    use_amp = bool(TARGET_CFG.get("use_amp", False)) and DEVICE.type == "cuda"
    scaler = torch.cuda.amp.GradScaler(enabled=use_amp)
    history = {"loss": [], "accuracy": [], "val_loss": [], "val_accuracy": [], "lr": []}
    best_state = None
    best_val_loss = float("inf")
    epochs_since_improved = 0

    for epoch in range(epochs):
        model.train()
        running_loss = 0.0
        total = 0
        correct = 0
        for inputs, labels in target_train_loader:
            inputs = inputs.to(DEVICE).float()
            labels = labels.to(DEVICE).long()
            optimizer.zero_grad(set_to_none=True)
            with torch.cuda.amp.autocast(enabled=use_amp):
                outputs = model(inputs)
                loss = criterion(outputs, labels)
            scaler.scale(loss).backward()
            scaler.step(optimizer)
            scaler.update()
            running_loss += loss.item()
            total += labels.size(0)
            correct += (outputs.argmax(dim=1) == labels).sum().item()

        epoch_loss = running_loss / max(len(target_train_loader), 1)
        epoch_acc = 100.0 * correct / max(total, 1)
        val_loss, val_acc = evaluate_fn(target_test_loader, model, criterion, DEVICE)
        _step_target_scheduler(scheduler, val_loss)
        current_lr = float(optimizer.param_groups[0]["lr"])
        history["loss"].append(float(epoch_loss))
        history["accuracy"].append(float(epoch_acc))
        history["val_loss"].append(float(val_loss))
        history["val_accuracy"].append(float(val_acc))
        history["lr"].append(current_lr)
        print(
            f"Epoch {epoch + 1}/{epochs}: train_loss={epoch_loss:.4f}, train_acc={epoch_acc:.2f}%, "
            f"val_loss={val_loss:.4f}, val_acc={val_acc:.2f}%, lr={current_lr:.2e}"
        )
        if val_loss + 1e-5 < best_val_loss:
            best_val_loss = val_loss
            best_state = deepcopy(model.state_dict())
            epochs_since_improved = 0
        else:
            epochs_since_improved += 1
            if patience and epochs_since_improved >= patience:
                print(f"Stopping early at epoch {epoch + 1} after {patience} epochs without improvement.")
                break

    if best_state is not None:
        model.load_state_dict(best_state)
    test_loss, test_acc = evaluate_fn(target_test_loader, model, criterion, DEVICE)
    summary = {
        "training_mode": "centralized",
        "model": "MobileNetV3Transfer",
        "dataset": data_config.get("dataset_name"),
        "history": history,
        "final_loss": float(test_loss),
        "final_accuracy": float(test_acc),
        "epochs_completed": len(history["loss"]),
        "batch_size": TARGET_BATCH_SIZE,
        "optimizer": deepcopy(target_optimizer_config),
        "scheduler": deepcopy(target_scheduler_config),
        "criterion": deepcopy(TARGET_CFG.get("criterion", {})),
        "use_amp": use_amp,
    }
    return model, summary


## 5. Train and Save the V3 Target

Run centralized target training and save both metrics and the checkpoint that seeds the attack notebook.


In [ ]:
target_model, target_summary = train_target_model()

print(
    f"Centralized V3 target metrics -> loss: {target_summary['final_loss']:.4f}, "
    f"accuracy: {target_summary['final_accuracy']:.2f}%"
)

target_summary


### Save and Validate the Target

Save `module4_target_training.json` and the MobileNetV3 checkpoint. The checkpoint name is configurable through `artifacts.target_checkpoint`.


In [ ]:
target_metrics_path = artifact_path("target_metrics", "module4_target_training.json")
with target_metrics_path.open("w") as f:
    json.dump(target_summary, f, indent=2)

target_checkpoint_path = artifact_path("target_checkpoint", "module4_v3_target.pt")
torch.save(target_model.state_dict(), target_checkpoint_path)

acc = float(target_summary.get("final_accuracy", 0.0))
loss = float(target_summary.get("final_loss", float("nan")))
history_len = int(target_summary.get("epochs_completed", 0))
if not np.isfinite(loss):
    raise ValueError("Target loss is not finite; revisit training settings.")
if history_len <= 0:
    raise ValueError("Target training history is empty.")
if acc < 5.0:
    raise ValueError(f"Target accuracy {acc:.2f}% is suspiciously low; revisit training settings.")

print(f"Target test accuracy: {acc:.2f}%")
print(f"Saved details to {target_metrics_path.resolve()}")
print(f"Saved checkpoint to {target_checkpoint_path.resolve()}")


### Target Training Curves

Plot the centralized target training and validation curves. These curves explain the V3 checkpoint quality before it is used by `attack_module.ipynb`.


In [ ]:
def plot_target_history(summary: dict) -> None:
    history = summary.get("history", {})
    if not history.get("loss"):
        print("No target history to plot.")
        return
    epochs = range(1, len(history["loss"]) + 1)
    fig, axes = plt.subplots(1, 2, figsize=(10, 4))
    axes[0].plot(epochs, history["loss"], marker="o", label="train")
    axes[0].plot(epochs, history.get("val_loss", []), marker="s", label="val")
    axes[0].set_title("V3 target loss")
    axes[0].set_xlabel("Epoch")
    axes[0].set_ylabel("Loss")
    axes[0].legend()
    axes[0].grid(True, alpha=0.3)

    axes[1].plot(epochs, history["accuracy"], marker="o", label="train")
    axes[1].plot(epochs, history.get("val_accuracy", []), marker="s", label="val")
    axes[1].set_title("V3 target accuracy")
    axes[1].set_xlabel("Epoch")
    axes[1].set_ylabel("Accuracy (%)")
    axes[1].legend()
    axes[1].grid(True, alpha=0.3)
    plt.tight_layout()
    out_path = artifact_path("target_history_plot", "target_training_history.png")
    plt.savefig(out_path, dpi=150)
    print(f"Saved {out_path.resolve()}")


plot_target_history(target_summary)


## Handoff Artifacts

After this notebook runs, `attack_module.ipynb` expects the target checkpoint in `artifacts/`.

| Artifact | Used by |
| --- | --- |
| `module4_config_used.json` | All split notebooks for reproducibility checks |
| `module4_target_training.json` | Target quality check before attacks |
| `module4_v3_target.pt` | Transfer evaluation and clean-vs-attacked FedAvg initialization |
| `target_training_history.png` | Instructor/student target training inspection |
